# LangGraph Demo: Agentic PEMDAS

This notebook demonstrates a small agentic loop using LangGraph.

The task is intentionally simple: solve arithmetic expressions like:

```text
2 + 3 * 6 - 4
```

The point is not to build a production calculator. The point is to show the agent pattern clearly:

```text
state -> plan -> act -> update state -> repeat
```

The agent has two tools:

- `MULT(a, b)` for multiplication
- `ADD(a, b)` for addition and subtraction

Subtraction is handled as `ADD(a, -b)`.

This keeps the demo small while still showing planning, tool use, state updates, and looping.

## Why this is a better LangGraph demo

A fixed workflow like this:

```text
input -> step 1 -> step 2 -> output
```

usually does not need LangGraph.

LangGraph becomes more useful when the system needs to decide what to do next based on current state.

This demo has:

- a planner node that chooses the next operation
- a tool node that executes the chosen tool
- an update node that rewrites the expression
- a conditional edge that loops until the expression is solved

## Graph shape

```mermaid
flowchart TD
    START --> planner
    planner --> tool
    tool --> update_expression
    update_expression --> planner
    planner --> END
```

The important part is the loop:

```text
planner -> tool -> update_expression -> planner
```

The graph keeps running until the planner says the expression is complete.

In [1]:
# If needed:
# pip install langgraph==0.3.0 pydantic==2.10.6

In [2]:
from __future__ import annotations

from typing import TypedDict, Literal, Optional
from dataclasses import dataclass
import re

from langgraph.graph import StateGraph, START, END

USE_LLM = True

In [3]:
from openai import OpenAI
import os
import json
from dotenv import load_dotenv
from getpass import getpass

# Try loading from .env if not already set
if not os.environ.get("OPENAI_API_KEY"):
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass  # dotenv is optional

# Final fallback: prompt user
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


## Define the graph state

The state is the shared memory that moves through the graph.

We keep it deliberately visible:

- `expression`: the current expression being simplified
- `next_tool`: the next tool to call
- `left` and `right`: tool arguments
- `result`: tool result
- `step_log`: human-readable trace of what happened
- `done`: whether the expression is solved

In [4]:
class PemdasState(TypedDict):
    original_expression: str
    expression: str
    next_tool: Optional[Literal["MULT", "ADD"]]
    left: Optional[int]
    right: Optional[int]
    result: Optional[int]
    done: bool
    step_log: list[str]

## Define the tools

These are intentionally boring. That is good for teaching.

The interesting part is not the tool implementation. The interesting part is how the graph decides which tool to call and how it updates state.

In [5]:
def MULT(a: int, b: int) -> int:
    return a * b


def ADD(a: int, b: int) -> int:
    return a + b


## A tiny parser for simple expressions

To keep the demo focused, this parser supports expressions made of integers and these operators:

```text
+  -  *
```

We skip division for the first version because integer vs floating-point behavior adds distraction.

Examples:

```text
2 + 3 * 6 - 4
10 - 2 * 3 + 1
```

In [6]:
def tokenize(expression: str) -> list[str]:
    '''Split a simple arithmetic expression into tokens.'''
    return re.findall(r"\d+|[+*\-]", expression.replace(" ", ""))

def format_tokens(tokens: list[str]) -> str:
    '''Format tokens back into a readable expression.'''
    return " ".join(tokens)


def is_number(token: str) -> bool:
    '''Verify string is just a number'''
    return re.fullmatch(r"\d+", token) is not None

def parse_llm_json(content: str) -> dict:
    '''Clean up JSON code generated by LLM by removing code fences, etc.'''
    content = content.strip()

    # Remove Markdown code fences if the model adds them.
    content = re.sub(r"^```json\s*", "", content)
    content = re.sub(r"^```\s*", "", content)
    content = re.sub(r"\s*```$", "", content)

    return json.loads(content)


def expression_is_solved(expression: str) -> bool:
    '''Check that the expression is just a number -- then it's done.'''
    tokens = tokenize(expression)
    return len(tokens) == 1 and is_number(tokens[0])

## Planner node

The planner decides what to do next.

This is where the agentic behavior appears:

1. If multiplication exists, do the first multiplication.
2. Otherwise, do the first addition or subtraction from left to right.
3. If only one number remains, finish.

This is deterministic planning, not LLM planning. That is intentional for the first teaching demo because students can focus on the graph loop.

In [7]:
def planner(state: PemdasState) -> dict:
    tokens = tokenize(state["expression"])

    # Done when the expression is just one number.
    if len(tokens) == 1 and is_number(tokens[0]):
        return {
            "done": True,
            "next_tool": None,
            "left": None,
            "right": None,
            "step_log": state["step_log"] + [f"Done: {tokens[0]}"],
        }

    # PEMDAS: multiplication before addition/subtraction.
    for i, token in enumerate(tokens):
        if token == "*":
            left = int(tokens[i - 1])
            right = int(tokens[i + 1])
            return {
                "done": False,
                "next_tool": "MULT",
                "left": left,
                "right": right,
                "step_log": state["step_log"] + [f"Plan: apply MULT({left}, {right})"],
            }

    # Then evaluate + and - from left to right.
    for i, token in enumerate(tokens):
        if token in ["+", "-"]:
            left = int(tokens[i - 1])
            raw_right = int(tokens[i + 1])
            right = raw_right if token == "+" else -raw_right

            return {
                "done": False,
                "next_tool": "ADD",
                "left": left,
                "right": right,
                "step_log": state["step_log"] + [f"Plan: apply ADD({left}, {right})"],
            }

    raise ValueError(f"Could not plan next step for expression: {state['expression']}")

In [8]:
def llm_planner(state: PemdasState) -> dict:
    planner_prompt = f"""
        You are a PEMDAS planning agent.

        Your job is to choose the next single tool call needed to simplify the expression.

        Available tools:
        - MULT(a, b): multiply two numbers
        - ADD(a, b): add two numbers. For subtraction, use ADD(a, -b).

        Rules:
        - Choose exactly one next tool call.
        - Multiplication has priority over addition/subtraction.
        - Addition and subtraction are handled left to right.
        - Return done=true only when the expression is a single number.
        - Do not solve the whole expression yourself.
        - Return JSON only.

        Expression:
        {state["expression"]}

        Return format:
        {{
        "done": false,
        "tool": "MULT",
        "left": 3,
        "right": 6
        }}
        """

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": planner_prompt}],
        temperature=0
    )

    content = response.choices[0].message.content

    try:
        decision = parse_llm_json(content)
    except:
        raise ValueError(f"Bad LLM output: {content}")

    if decision["done"]:
        if expression_is_solved(state["expression"]):
            return {
                "done": True,
                "next_tool": None,
                "left": None,
                "right": None,
                "step_log": state["step_log"] + ["LLM: done"],
            }
        else:
            return {
                "done": False,
                "next_tool": "MULT" if "*" in state["expression"] else "ADD",
                "left": None,
                "right": None,
                "step_log": state["step_log"] + [
                    "LLM incorrectly said done; falling back to deterministic planner"
                ],
            }

    return {
        "done": False,
        "next_tool": decision["tool"],
        "left": decision["left"],
        "right": decision["right"],
        "step_log": state["step_log"] + [
            f"LLM Plan: {decision['tool']}({decision['left']}, {decision['right']})"
        ],
    }

## Tool node

The tool node executes the selected tool.

In a larger agent, this could be a search tool, database tool, calendar tool, CRM tool, or code execution tool. Here it is just arithmetic.

In [9]:
def tool_node(state: PemdasState) -> dict:
    tool = state["next_tool"]
    left = state["left"]
    right = state["right"]

    if tool == "MULT":
        result = MULT(left, right)
    elif tool == "ADD":
        result = ADD(left, right)
    else:
        raise ValueError(f"Unknown tool: {tool}")

    return {
        "result": result,
        "step_log": state["step_log"] + [f"Tool: {tool}({left}, {right}) = {result}"],
    }

## Update node

The update node rewrites the expression by replacing the operation that was just evaluated.

Example:

```text
2 + 3 * 6 - 4
```

After `MULT(3, 6) = 18`, it becomes:

```text
2 + 18 - 4
```

In [10]:
def update_expression(state: PemdasState) -> dict:
    tokens = tokenize(state["expression"])
    tool = state["next_tool"]
    result = state["result"]

    if tool == "MULT":
        target_ops = ["*"]
    elif tool == "ADD":
        target_ops = ["+", "-"]
    else:
        raise ValueError(f"Unknown tool: {tool}")

    for i, token in enumerate(tokens):
        if token in target_ops:
            if tool == "ADD":
                # Make sure this is the same operation the planner selected.
                left = int(tokens[i - 1])
                raw_right = int(tokens[i + 1])
                right = raw_right if token == "+" else -raw_right
                if left != state["left"] or right != state["right"]:
                    continue

            if tool == "MULT":
                left = int(tokens[i - 1])
                right = int(tokens[i + 1])
                if left != state["left"] or right != state["right"]:
                    continue

            new_tokens = tokens[: i - 1] + [str(result)] + tokens[i + 2 :]
            # Example
            # Input: 2 + 3 - 5
            # Planner: Operation "+": left=2, right=3,
            # ADD tool: return 5
            # update_expression: new_tokens = [] + [5] + [-, 5] = [5,'-',5]
            # new_expression: "5 - 5"


            new_expression = format_tokens(new_tokens)

            return {
                "expression": new_expression,
                "step_log": state["step_log"] + [f"Update: {new_expression}"],
            }

    raise ValueError("Could not update expression.")

## Conditional routing

After planning, the graph either:

- ends if the expression is solved
- or calls the selected tool

In [11]:
def should_continue(state: PemdasState) -> Literal["continue", "end"]:
    return "end" if state["done"] else "continue"

## Build the LangGraph workflow

In [12]:
mdas_workflow = StateGraph(PemdasState)

if USE_LLM:
    mdas_workflow.add_node("planner", llm_planner)
else:
    mdas_workflow.add_node("planner", planner)

mdas_workflow.add_node("tool", tool_node)
mdas_workflow.add_node("update_expression", update_expression)

mdas_workflow.add_edge(START, "planner")

mdas_workflow.add_conditional_edges(
    "planner",
    should_continue,
    {
        "continue": "tool",
        "end": END,
    },
)

mdas_workflow.add_edge("tool", "update_expression")
mdas_workflow.add_edge("update_expression", "planner")

mdas_graph = mdas_workflow.compile()

## Visualize the graph

This is the simple teaching version.

In [13]:
from IPython.display import Markdown
from IPython.display import Markdown, display

mdas_mermaid = mdas_graph.get_graph().draw_mermaid()
display(Markdown(f"```mermaid\n{mdas_mermaid}\n```"))

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	planner(planner)
	tool(tool)
	update_expression(update_expression)
	__end__([<p>__end__</p>]):::last
	__start__ --> planner;
	planner -. &nbsp;end&nbsp; .-> __end__;
	planner -. &nbsp;continue&nbsp; .-> tool;
	tool --> update_expression;
	update_expression --> planner;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

## Run the agent

In [14]:
initial_state: PemdasState = {
    "original_expression": "2 + 3 * 6 - 4",
    "expression": "2 + 3 * 6 - 4",
    "next_tool": None,
    "left": None,
    "right": None,
    "result": None,
    "done": False,
    "step_log": [],
}

final_state = mdas_graph.invoke(initial_state)
final_state

{'original_expression': '2 + 3 * 6 - 4',
 'expression': '16',
 'next_tool': None,
 'left': None,
 'right': None,
 'result': 16,
 'done': True,
 'step_log': ['LLM Plan: MULT(3, 6)',
  'Tool: MULT(3, 6) = 18',
  'Update: 2 + 18 - 4',
  'LLM Plan: ADD(2, 18)',
  'Tool: ADD(2, 18) = 20',
  'Update: 20 - 4',
  'LLM Plan: ADD(20, -4)',
  'Tool: ADD(20, -4) = 16',
  'Update: 16',
  'LLM: done']}

## Show state progress clearly

This is the most important teaching output. It shows the loop and state changes.

In [15]:
for step in final_state["step_log"]:
    print(step)

print()
print("Original:", final_state["original_expression"])
print("Final:", final_state["expression"])

LLM Plan: MULT(3, 6)
Tool: MULT(3, 6) = 18
Update: 2 + 18 - 4
LLM Plan: ADD(2, 18)
Tool: ADD(2, 18) = 20
Update: 20 - 4
LLM Plan: ADD(20, -4)
Tool: ADD(20, -4) = 16
Update: 16
LLM: done

Original: 2 + 3 * 6 - 4
Final: 16


## Try a few more examples

In [16]:
examples = [
    "10 - 2 * 3 + 1",
    "4 * 5 + 6",
    "8 + 2 * 3 * 4 - 5",
]

for expr in examples:
    state: PemdasState = {
        "original_expression": expr,
        "expression": expr,
        "next_tool": None,
        "left": None,
        "right": None,
        "result": None,
        "done": False,
        "step_log": [],
    }

    result = mdas_graph.invoke(state)

    print("=" * 60)
    print("Expression:", expr)
    for step in result["step_log"]:
        print(step)
    print("Answer:", result["expression"])

Expression: 10 - 2 * 3 + 1
LLM Plan: MULT(2, 3)
Tool: MULT(2, 3) = 6
Update: 10 - 6 + 1
LLM Plan: ADD(10, -6)
Tool: ADD(10, -6) = 4
Update: 4 + 1
LLM Plan: ADD(4, 1)
Tool: ADD(4, 1) = 5
Update: 5
LLM: done
Answer: 5
Expression: 4 * 5 + 6
LLM Plan: MULT(4, 5)
Tool: MULT(4, 5) = 20
Update: 20 + 6
LLM Plan: ADD(20, 6)
Tool: ADD(20, 6) = 26
Update: 26
LLM: done
Answer: 26
Expression: 8 + 2 * 3 * 4 - 5
LLM Plan: MULT(2, 3)
Tool: MULT(2, 3) = 6
Update: 8 + 6 * 4 - 5
LLM Plan: MULT(6, 4)
Tool: MULT(6, 4) = 24
Update: 8 + 24 - 5
LLM Plan: ADD(8, 24)
Tool: ADD(8, 24) = 32
Update: 32 - 5
LLM Plan: ADD(32, -5)
Tool: ADD(32, -5) = 27
Update: 27
LLM: done
Answer: 27


## Teaching checkpoint

Ask students:

1. Where is the state stored?
2. Which node decides what happens next?
3. Which edge creates the loop?
4. Why would this be overkill for a fixed one-shot calculation?
5. What changes if the tools are external APIs instead of arithmetic functions?

The main lesson:

```text
LangGraph is useful when the application needs controlled iteration over changing state.
```

## Extension - use sub-graphs to support full PEMDAS 

In [17]:

def find_innermost_parentheses(expr: str):
    match = re.search(r"\(([^()]+)\)", expr)
    if not match:
        return None
    return match.group(1)

def solve_parentheses_node(state):
    inner = find_innermost_parentheses(state["expression"])

    sub_state = {
        "original_expression": inner,
        "expression": inner,
        "next_tool": None,
        "left": None,
        "right": None,
        "result": None,
        "done": False,
        "step_log": [],
    }

    result = mdas_graph.invoke(sub_state)

    new_expr = state["expression"].replace(
        f"({inner})",
        result["expression"],
        1
    )

    return {
        "expression": new_expr,
        "step_log": state["step_log"] + [
            f"Subgraph solved ({inner}) -> {result['expression']}",
            f"Update: {new_expr}",
        ],
    }

In [18]:
from typing_extensions import Literal
from langgraph.graph import StateGraph, START, END

def pemdas_planner(state: PemdasState) -> dict:
    # Parentheses have highest priority.
    if find_innermost_parentheses(state["expression"]) is not None:
        return {
            "done": False,
            "next_tool": "PARENS",
            "left": None,
            "right": None,
            "result": None,
            "step_log": state["step_log"] + ["Plan: solve innermost parentheses"],
        }

    # Otherwise use the existing MDAS planner.
    return llm_planner(state)


def pemdas_route(state: PemdasState) -> Literal["parentheses", "tool", "end"]:
    if state["done"]:
        return "end"

    if state["next_tool"] == "PARENS":
        return "parentheses"

    return "tool"

In [19]:
pemdas_workflow = StateGraph(PemdasState)

pemdas_workflow.add_node("planner", pemdas_planner)
pemdas_workflow.add_node("solve_parentheses", solve_parentheses_node)
pemdas_workflow.add_node("tool", tool_node)
pemdas_workflow.add_node("update_expression", update_expression)

pemdas_workflow.add_edge(START, "planner")

pemdas_workflow.add_conditional_edges(
    "planner",
    pemdas_route,
    {
        "parentheses": "solve_parentheses",
        "tool": "tool",
        "end": END,
    },
)

pemdas_workflow.add_edge("solve_parentheses", "planner")
pemdas_workflow.add_edge("tool", "update_expression")
pemdas_workflow.add_edge("update_expression", "planner")

pemdas_graph = pemdas_workflow.compile()

In [20]:
pemdas_mermaid = pemdas_graph.get_graph().draw_mermaid()
display(Markdown(f"```mermaid\n{pemdas_mermaid}\n```"))

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	planner(planner)
	solve_parentheses(solve_parentheses)
	tool(tool)
	update_expression(update_expression)
	__end__([<p>__end__</p>]):::last
	__start__ --> planner;
	planner -. &nbsp;end&nbsp; .-> __end__;
	planner -. &nbsp;parentheses&nbsp; .-> solve_parentheses;
	planner -.-> tool;
	solve_parentheses --> planner;
	tool --> update_expression;
	update_expression --> planner;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [23]:
my_expression = "(2 * (3 + 4) + 7) * 6"

state = {
    "original_expression": my_expression,
    "expression": my_expression,
    "next_tool": None,
    "left": None,
    "right": None,
    "result": None,
    "done": False,
    "step_log": [],
}

result = pemdas_graph.invoke(state)
print(state["original_expression"])
for step in result["step_log"]:
    print(step)

print("Final:", result["expression"])

(2 * (3 + 4) + 7) * 6
Plan: solve innermost parentheses
Subgraph solved (3 + 4) -> 7
Update: (2 * 7 + 7) * 6
Plan: solve innermost parentheses
Subgraph solved (2 * 7 + 7) -> 21
Update: 21 * 6
LLM Plan: MULT(21, 6)
Tool: MULT(21, 6) = 126
Update: 126
LLM: done
Final: 126
